In [38]:
!pip install requests beautifulsoup4 pandas

In [39]:
#Importing required libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import sqlite3
import os

In [40]:
#Scraping the books
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

books = []

for page in range(1, 5):

    url = BASE_URL.format(page)

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):

        if len(books) ==70:
            break

        # Title
        title_tag = book.select_one("h3 a")
        title = title_tag["title"]

        # Price
        price = book.select_one(".price_color").get_text(strip=True)

        # Rating
        star_rating = book.select_one(".star-rating")["class"][1]

        # Availability
        availability = book.select_one(
            ".availability"
        ).get_text(" ", strip=True)

        # Book detail page
        book_url = urljoin(
            url,
            title_tag["href"]
        )

        detail_response = requests.get(
            book_url,
            timeout=10
        )
        detail_response.raise_for_status()

        detail_soup = BeautifulSoup(
            detail_response.text,
            "html.parser"
        )

        # Category
        breadcrumb = detail_soup.select(
            "ul.breadcrumb li"
        )

        category = breadcrumb[-2].get_text(
            strip=True
        )

        books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

    if len(books) ==70:
        break

print("Total books scraped:", len(books))

Total books scraped: 70


In [33]:
#Creating DataFrame for scraped books
df = pd.DataFrame(books)

df.head()

,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


In [41]:
#Checking the number of categories
print("Books:", len(df))
print("Categories:", df["category"].nunique())

Books: 70
Categories: 27


In [43]:

df = pd.DataFrame(books)

# Convert all scraped values to strings
df["price"] = df["price"].astype(str)
df["star_rating"] = df["star_rating"].astype(str)
df["availability"] = df["availability"].astype(str)

# 1. Clean price
# Remove Â£ symbol and convert to number
df["price_gbp"] = (
    df["price"]
    .str.replace("Â£", "", regex=False)
    .str.strip()
)

df["price_gbp"] = pd.to_numeric(
    df["price_gbp"],
    errors="coerce"
)

# If any price is invalid, replace it with the median price
df["price_gbp"] = df["price_gbp"].fillna(
    df["price_gbp"].median()
)

# 2. Convert star rating to numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

# Remove rows where rating could not be converted
df = df.dropna(subset=["rating"])

df["rating"] = df["rating"].astype(int)

# 3. Convert availability to True/False
df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

# 4. Convert GBP to INR
# Fixed conversion rate required by the project
GBP_TO_INR = 105.50

df["price_inr"] = (
    df["price_gbp"] * GBP_TO_INR
).round(2)

# 5. Keep only required columns
df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].reset_index(drop=True)

# Checking results

print("Cleaned Data:")
display(df.head(10))

print("\nTotal Books:", len(df))
print("Total Categories:", df["category"].nunique())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

Cleaned Data:


,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.74,3,True,Poetry
1,Tipping the Velvet,53.74,5669.57,1,True,Historical Fiction
2,Soumission,50.10,5285.55,1,True,Fiction
3,Sharp Objects,47.82,5045.01,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History
5,The Requiem Red,22.65,2389.57,1,True,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,True,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,1891.62,3,True,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,2384.30,4,True,Default
9,The Black Maria,52.15,5501.82,1,True,Poetry



Total Books: 70
Total Categories: 27

Missing Values:
title        0
price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64

Data Types:
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [44]:
#Creating normalized SQLite schema
import sqlite3
import os

# Create database
db_name = "zepto_books.db"

# Remove old database if it exists
if os.path.exists(db_name):
    os.remove(db_name)

conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Enable foreign keys
cursor.execute("PRAGMA foreign_keys = ON")

# Create categories table
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

# Create books table
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("SQLite database created successfully.")
print("Tables created: categories, books")

SQLite database created successfully.
Tables created: categories, books


In [45]:
#Insert data + execute SQL queries + save outputs

# INSERT DATA INTO SQLITE

# Insert unique categories
for category in df["category"].dropna().unique():

    cursor.execute(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

conn.commit()


# Insert books
for _, row in df.iterrows():

    # Get category ID
    cursor.execute(
        """
        SELECT category_id
        FROM categories
        WHERE category_name = ?
        """,
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            float(row["price_gbp"]),
            float(row["price_inr"]),
            int(row["rating"]),
            int(row["in_stock"]),
            category_id
        )
    )

conn.commit()

# SQL QUERIES

# 1. SELECT + WHERE
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4
"""

# 2. ORDER BY
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
"""

# 3. LIMIT
query3 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

# 4. DISTINCT
query4 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating
"""

# 5. BETWEEN
query5 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp
"""

# 6. JOIN
query6 = """
SELECT
    c.category_name,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.title
"""

# Execute queries

result1 = pd.read_sql(query1, conn)
result2 = pd.read_sql(query2, conn)
result3 = pd.read_sql(query3, conn)
result4 = pd.read_sql(query4, conn)
result5 = pd.read_sql(query5, conn)
result6 = pd.read_sql(query6, conn)

# Display results

print("Query 1 - SELECT + WHERE")
display(result1.head(10))

print("Query 2 - ORDER BY")
display(result2.head(10))

print("Query 3 - LIMIT")
display(result3)

print("Query 4 - DISTINCT")
display(result4)

print("Query 5 - BETWEEN")
display(result5.head(10))

print("Query 6 - JOIN")
display(result6.head(10))

# Save query string and output
os.makedirs("sql_outputs", exist_ok=True)

query_results = {
    "select_where": result1,
    "order_by": result2,
    "limit": result3,
    "distinct": result4,
    "between": result5,
    "join": result6
}

for name, result in query_results.items():
    result.to_csv(
        f"sql_outputs/{name}.csv",
        index=False
    )


queries = {
    "SELECT_WHERE": query1,
    "ORDER_BY": query2,
    "LIMIT": query3,
    "DISTINCT": query4,
    "BETWEEN": query5,
    "JOIN": query6
}

with open("sql_outputs/queries.sql", "w") as file:

    for name, query in queries.items():
        file.write(f"-- {name}\n")
        file.write(query.strip())
        file.write("\n\n")


print("======================================")
print("Data inserted successfully")
print("Books:", len(df))
print("SQL queries executed: 6")
print("Query outputs saved")
print("Query strings saved")
print("======================================")

Query 1 - SELECT + WHERE


,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4
5,Set Me Free,17.46,5
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
7,Rip it Up and Start Again,35.02,5
8,Chase Me (Paris Nights #2),25.27,5
9,Black Dust,34.53,5


Query 2 - ORDER BY


,title,price_gbp,rating
0,The Death of Humanity: and the Case for Life,58.11,4
1,Slow States of Collapse: Poems,57.31,3
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3
3,The Past Never Ends,56.50,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
5,The Secret of Dreadwillow Carse,56.13,1
6,The Electric Pencil: Drawings from Inside Stat...,56.06,1
7,Birdsong: A Story in Pictures,54.64,3
8,Sapiens: A Brief History of Humankind,54.23,5
9,The Murder That Never Was (Forensic Instincts #5),54.11,3


Query 3 - LIMIT


...

Query 5 - BETWEEN


,title,price_gbp,price_inr
0,The Inefficiency Assassin: Time Management Tac...,20.59,2172.24
1,Shakespeare's Sonnets,20.66,2179.63
2,America's Cradle of Quarterbacks: Western Penn...,22.50,2373.75
3,The Boys in the Boat: Nine Americans and Their...,22.60,2384.30
4,The Requiem Red,22.65,2389.57
5,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10
6,The Elephant Tree,23.82,2513.01
7,Olio,23.88,2519.34
8,The Mindfulness and Acceptance Workbook for An...,23.89,2520.40
9,Chase Me (Paris Nights #2),25.27,2665.98


Query 6 - JOIN


,category_name,title,price_gbp,price_inr,rating,in_stock
0,Nonfiction,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,1
1,Romance,Black Dust,34.53,3642.92,5,1
2,Romance,Chase Me (Paris Nights #2),25.27,2665.98,5,1
3,Fiction,Private Paris (Private #10),47.61,5022.85,5,1
4,Music,Rip it Up and Start Again,35.02,3694.61,5,1
5,History,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1
6,Sequential Art,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,1
7,Young Adult,Set Me Free,17.46,1842.03,5,1
8,Philosophy,Sophie's World,15.94,1681.67,5,1
9,Thriller,The Elephant Tree,23.82,2513.01,5,1


Data inserted successfully
Books: 70
SQL queries executed: 6
Query outputs saved
Query strings saved


In [47]:
# Merging
# Read SQL tables into.pandas
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

# Read at least two query results using pd.read_sql()
sql_result_1 = pd.read_sql(
    query1,
    conn
)

sql_result_2 = pd.read_sql(
    query5,
    conn
)

print("First pd.read_sql() result:")
display(sql_result_1.head())

print("Second pd.read_sql() result:")
display(sql_result_2.head())

# Reproduce JOIN using pd.merge()
pandas_join = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

# Keep same columns as SQL JOIN
pandas_join = pandas_join[
    [
        "category_name",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock"
    ]
]

# Same sorting as SQL JOIN
pandas_join = pandas_join.sort_values(
    ["rating", "title"],
    ascending=[False, True]
).reset_index(drop=True)

# Compare SQL JOIN and pandas merge
sql_join = result6.reset_index(drop=True)

print("SQL JOIN result:")
display(sql_join.head(10))

print("Pandas merge result:")
display(pandas_join.head(10))


# Check if both results are equivalent
print(
    "SQL JOIN and Pandas merge are equivalent:",
    sql_join.equals(pandas_join)
)



First pd.read_sql() result:


,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4


Second pd.read_sql() result:


,title,price_gbp,price_inr
0,The Inefficiency Assassin: Time Management Tac...,20.59,2172.24
1,Shakespeare's Sonnets,20.66,2179.63
2,America's Cradle of Quarterbacks: Western Penn...,22.50,2373.75
3,The Boys in the Boat: Nine Americans and Their...,22.60,2384.30
4,The Requiem Red,22.65,2389.57


SQL JOIN result:


,category_name,title,price_gbp,price_inr,rating,in_stock
0,Nonfiction,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,1
1,Romance,Black Dust,34.53,3642.92,5,1
2,Romance,Chase Me (Paris Nights #2),25.27,2665.98,5,1
3,Fiction,Private Paris (Private #10),47.61,5022.85,5,1
4,Music,Rip it Up and Start Again,35.02,3694.61,5,1
5,History,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1
6,Sequential Art,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,1
7,Young Adult,Set Me Free,17.46,1842.03,5,1
8,Philosophy,Sophie's World,15.94,1681.67,5,1
9,Thriller,The Elephant Tree,23.82,2513.01,5,1


Pandas merge result:


,category_name,title,price_gbp,price_inr,rating,in_stock
0,Nonfiction,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,1
1,Romance,Black Dust,34.53,3642.92,5,1
2,Romance,Chase Me (Paris Nights #2),25.27,2665.98,5,1
3,Fiction,Private Paris (Private #10),47.61,5022.85,5,1
4,Music,Rip it Up and Start Again,35.02,3694.61,5,1
5,History,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1
6,Sequential Art,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,1
7,Young Adult,Set Me Free,17.46,1842.03,5,1
8,Philosophy,Sophie's World,15.94,1681.67,5,1
9,Thriller,The Elephant Tree,23.82,2513.01,5,1


SQL JOIN and Pandas merge are equivalent: True
